# Multilingual Health QA in Low-Resource African Languages
### Unified end-to-end notebook (EDA → retrieval → fine-tuning → submission)

**Governing insight:** the metric is `0.37·ROUGE-1 + 0.37·ROUGE-L + 0.26·LLM-judge` → **74% is
lexical overlap**, so the system is **retrieval-first**, with fine-tuning layered on the configs
where retrieval is structurally capped (≈99%-unique answers). Best public score: **0.598**.

Run cells in order. Use a **GPU** runtime for the fine-tuning section (Runtime → Change runtime type → T4).

## 1. Setup

In [1]:
!git clone https://github.com/<your-username>/hash-multilingual-health-qa.git
%cd hash-multilingual-health-qa
!pip install -q -r requirements.txt
import sys; sys.path.append('src')
print('setup complete')

zsh:1: no such file or directory: your-username
[Errno 2] No such file or directory: 'hash-multilingual-health-qa'
/Users/kelvinrwihimba/Documents/Final Summative/QA_Summative_ML/notebooks
zsh:1: command not found: pip


/Users/kelvinrwihimba/Library/Python/3.9/lib/python/site-packages/IPython/core/magics/osm.py:393: UserWarning: using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


setup complete


## 2. Data
Upload the challenge files `Train.csv`, `Val.csv`, `Test.csv` (not redistributed in the repo).

In [2]:
import os
os.makedirs('Data', exist_ok=True)
from google.colab import files
up = files.upload()  # Train.csv, Val.csv, Test.csv
for n in up:
    if n.endswith('.csv') and n != 'SampleSubmission.csv':
        os.replace(n, f'Data/{n}')
print('Data/:', os.listdir('Data'))

ModuleNotFoundError: No module named 'google'

## 3. Exploratory Data Analysis
The decisive variable is **answer uniqueness** per subset.

In [ ]:
import pandas as pd
tr = pd.read_csv('Data/Train.csv'); va = pd.read_csv('Data/Val.csv'); te = pd.read_csv('Data/Test.csv')
print('shapes', tr.shape, va.shape, te.shape)
dup = (tr.groupby('subset').output.nunique()/tr.groupby('subset').size()).sort_values()
print('\nanswer uniqueness (unique/rows):'); print(dup.round(3))
print('\ntest mix %:'); print((te.subset.value_counts(normalize=True)*100).round(1))

In [ ]:
!python src/make_figures.py
from IPython.display import Image, display
for f in ['fig1_subset_distribution.png','fig2_answer_dup_rate.png','fig3_answer_length.png']:
    display(Image(f'reports/figures/{f}'))

## 4. Retrieval pipeline
Baseline → per-subset routing + global bank → MPNet rerank → Ghana hybrid.
Each step is Val-gated. The MPNet encoder downloads once (~1 GB).

In [ ]:
# 4a. Baseline floor (CPU): per-language char_wb TF-IDF nearest-answer
!python src/baseline_retrieval.py

In [ ]:
# 4b. Best retrieval submission: rerank + Ghana hybrid (+ Val-bank for Test)
#     prints the honest Train-only Val scoreboard, writes submissions/submission_v4.csv
!python src/make_submission_v4.py

In [ ]:
for f in ['fig4_experiment_progression.png','fig5_per_subset_v5.png',
          'fig6_retrieval_vs_oracle.png','fig7_calibration.png']:
    display(Image(f'reports/figures/{f}'))

## 5. Fine-tuning (GPU) — mT5 on the Ghana configs
Retrieval is capped on Ghana (oracle@20 ≈ 0.39). We fine-tune mT5-small (full + LoRA/PEFT) to
try to compose better answers. Verdict (gated on Val): generation converges but does **not** beat
retrieval on the bespoke answers, so retrieval is kept. ~15–25 min per run on a T4.

In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
import os; os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# 5a. mT5-small FULL fine-tune
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/train_generator.py \
    --data-dir Data --output-dir generator_outputs --run-name mt5small_full \
    --model-name google/mt5-small --subsets Aka_Gha,Eng_Gha \
    --epochs 2 --train-bs 16 --grad-accum 1 --lr 1e-3 \
    --max-input 96 --max-target 160 --gen-len 160 --beams 2 --seed 42

In [ ]:
# 5b. mT5-small LoRA / PEFT (run separately so the GPU frees between runs)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/train_generator.py \
    --data-dir Data --output-dir generator_outputs --run-name mt5small_lora \
    --model-name google/mt5-small --subsets Aka_Gha,Eng_Gha --use-lora \
    --epochs 2 --train-bs 16 --grad-accum 1 --lr 5e-4 \
    --max-input 96 --max-target 160 --gen-len 160 --beams 2 --seed 42

In [ ]:
# learning curves + generation-vs-retrieval
!python src/make_figures.py
for f in ['fig8_learning_curves.png','fig9_generation_vs_retrieval.png']:
    display(Image(f'reports/figures/{f}'))

## 6. Gate + final submission
Replace a Ghana answer with generation ONLY if it beats retrieval on that subset's Val.

In [ ]:
!python src/make_submission_v5.py --gen-dir generator_outputs/mt5small_full
import pandas as pd
s = pd.read_csv('submissions/submission_v5.csv')
print('final submission rows:', len(s), '| columns:', list(s.columns))

## 7. Results
| Stage | Local portion (/0.74) | Public |
|---|---:|---:|
| Baseline | 0.3150 | — |
| + routing + global bank | 0.3292 | — |
| + MPNet rerank | 0.3572 | 0.579 |
| **+ Ghana hybrid (best)** | **0.3701** | **0.598** |

Validated mapping **`public ≈ local + 0.096`**. The ceiling is set by answer uniqueness, not
model quality — the central, evidence-backed finding. Full analysis, the 14 documented
experiments, ethics and future work: `reports/REPORT.md` (and `REPORT.html`).